# OpenAI API 기초 (Colab 실습 노트)

이 노트는 **OpenAI API를 Colab에서 빠르게 실습**하기 위한 최소 구성을 제공합니다.  
핵심은 **Chat Completions 호출 → temperature 제어 → token(출력 길이) 제어 → 사용량(usage) 확인**입니다.

---

## 1. OpenAI API는 무엇인가?
- OpenAI API는 모델(GPT 등)을 **HTTP 요청으로 호출**해 텍스트 생성/요약/질의응답 등을 수행하게 하는 인터페이스입니다.
- 대화형 생성은 보통 **Chat Completions** 형식으로 `messages`(role/content) 배열을 전달합니다.

---

## 2. API Key 보안 (가장 중요)
- API Key는 **비밀값**이며, 브라우저/앱 같은 클라이언트 코드에 노출하면 안 됩니다.  
- 권장: **환경변수/시크릿 매니저**로 주입하고, 레포지토리에 절대 커밋하지 않습니다. :contentReference[oaicite:3]{index=3}

> Colab에서는 좌측 **Secrets**(키 아이콘)에 `OPENAI_API_KEY`를 저장해 사용하세요.




In [ ]:
#실습 준비
!pip -q install openai

In [28]:
import getpass
from openai import OpenAI

while True:
    key = getpass.getpass("OPENAI_API_KEY (must start with sk-): ").strip()
    if key.startswith("sk-"):
        break
    print("Not a key. Paste the real key starting with 'sk-' (not bullets/dots).")

client = OpenAI(api_key=key)
print(client.models.list().data[0].id)


OPENAI_API_KEY (must start with sk-): ··········
gpt-4-0613


In [ ]:
client.models.list().data

## 왜 `sk-proj-...` 인가?
- `sk-proj-...`는 **Project(프로젝트) 단위로 발급된 OpenAI API 키**라서 접두사가 그렇게 붙습니다. 정상입니다.

## 코드가 하는 일 (한 줄 요약)
1) `OPENAI_API_KEY` 환경변수에 **키 값**을 넣고  
2) 다시 `OPENAI_API_KEY`를 읽어서 `OpenAI(api_key=...)`에 전달한 뒤  
3) `models.list()`로 **연결이 정상인지 테스트**합니다.


- gpt-4-0613 : 인증/연결이 성공했고, 사용 가능한 모델 목록을 정상적으로 받아온 것
    - GPT-4의 2023-06-13 스냅샷(버전 고정 이름)
    - 날짜 접미사(0613)는 해당 날짜에 고정된 모델 릴리스(특히 당시 “function calling” 업데이트와 함께 소개된 스냅샷)를 뜻함.

In [ ]:
import os
from openai import OpenAI


# 2) 모델은 아까 확인된 걸 쓰는 게 안전 (예: gpt-4-0613)
MODEL = "gpt-4-0613"   # 필요하면 "gpt-4o"로 바꿔도 됨(접근권한 있을 때)

response = client.responses.create(
    model=MODEL,
    instructions="You are a concise coding assistant.",
    input="How do I check if a Python object is an instance of a class?",
)

print(response.output_text)


To check if a Python object is an instance of a specific class, you can use the built-in `isinstance()` function. The syntax is `isinstance(object, class)`. It returns `True` if the object is an instance of the class, and `False` otherwise. 

Here's an example:

```python
class MyClass:
  pass

x = MyClass()

print(isinstance(x, MyClass))  # Outputs: True
```


In [ ]:
import os
from openai import OpenAI


messages = [
    {"role": "system", "content": "You are a concise coding assistant."},
    {"role": "user", "content": "Explain Python isinstance() with one simple example."},
    {"role": "user", "content": "Now add one edge case involving inheritance."},
]

resp = client.responses.create(
    model=MODEL,
    input=messages,
)

print(resp.output_text)


In Python, `isinstance()` is a built-in function that checks if the specified object is an instance of the specified type. This function returns `True` if the object is an instance of the type, and `False` otherwise. 

Here is a simple example:

```python
num = 5
print(isinstance(num, int))  # Outputs: True
```

In this example, `isinstance()` checks if `num` is an instance of `int`. Since `num` is indeed an integer, the function returns `True`.

Now, let's consider an edge case with inheritance. In Python, `isinstance()` also considers inheritance, meaning that if a class B is derived from class A, an instance of B is considered an instance of A too.

For example:

```python
class Fruit:
  pass

class Apple(Fruit):
  pass

apple = Apple()
print(isinstance(apple, Fruit))  # Outputs: True
```

In this example, `Apple` is a subclass of `Fruit`, so an instance of `Apple` is considered an instance of `Fruit`, and `isinstance()` returns `True`.


In [ ]:
import os
from openai import OpenAI


messages = [
    {"role": "system", "content": "You are a concise coding assistant."},
    {"role": "user", "content": "안녕하세요! 저는 존이라고 합니다."},
    {"role": "assistant", "content": "안녕하세요. 존님! 만나서 반갑습니다. 오늘은 어떤 이야기를 나눌까요?"},
    {"role": "user","content": "제 이름을 알고 계신가요?"}
]

resp = client.responses.create(
    model=MODEL,
    input=messages,
)

print(resp.output_text)


네, 당신이 존님이라고 말씀하셨기 때문에 알고 있습니다.


In [ ]:
!pip install tiktoken

In [ ]:
import tiktoken
text = "LLM을 사용해 했던 것을 만들기 쉽지만, 프로덕션에서 사용할 수 있는 것들을 매우 어렵다"

encoding = tiktoken.encoding_for_model('gpt-4o')
tokens = encoding.encode(text)
print(len(tokens))

25


In [ ]:
#파라미터 조절로 답변 스타일 변화 보기

resp_low = client.responses.create(
    model=MODEL,
    input="Give 3 bullet points on isinstance().",
    temperature=0.2,
)
resp_high = client.responses.create(
    model=MODEL,
    input="Give 3 bullet points on isinstance().",
    temperature=1.0,
)

print("=== temperature=0.2 ===")
print(resp_low.output_text)
print("\n=== temperature=1.0 ===")
print(resp_high.output_text)

=== temperature=0.2 ===
- isinstance() is a built-in function in Python that checks if an object or variable is an instance or subclass of a specified class or type.
- It takes two parameters: the object or variable to be checked, and the class or type against which the check is to be performed. It returns True if the object is an instance or subclass of the specified class or type, and False otherwise.
- isinstance() can also check if an object is an instance of any one of several classes or types if given a tuple of classes or types as the second parameter. For example, isinstance(x, (int, float)) checks if x is an instance of either int or float.

=== temperature=1.0 ===
- The isinstance() function in Python checks if an object or variable is an instance or subclass of a specified class or type.
- This built-in function takes two parameters: the first one is the object or variable to check, and the second is the class or type to compare with. 
- It returns either True (if the object

In [ ]:
#파라미터 조절로 답변 스타일 변화 보기

resp_low = client.responses.create(
    model=MODEL,
    input="겨울에 여행가기 좋은 국내 명소",
    temperature=0.2,
)
resp_high = client.responses.create(
    model=MODEL,
    input="겨울에 여행가기 좋은 국내 명소",
    temperature=1.0,
)

print("=== temperature=0.2 ===")
print(resp_low.output_text)
print("\n=== temperature=1.0 ===")
print(resp_high.output_text)

=== temperature=0.2 ===
1. 강원도 평창: 겨울에는 특히 더 아름다운 평창은 겨울 스포츠를 즐길 수 있는 곳으로 유명합니다. 2018년 동계 올림픽이 열렸던 알펜시아 리조트, 용평 리조트 등에서 스키나 스노보드를 즐길 수 있습니다.

2. 경북 경주: 겨울에 눈이 내리면 더욱 분위기 있는 경주는 역사적인 명소를 찾는 여행객들에게 인기가 많습니다. 불국사, 첨성대, 안압지 등에서 겨울 풍경을 즐길 수 있습니다.

3. 제주도: 겨울에도 따뜻한 날씨를 자랑하는 제주도는 한라산의 눈꽃 풍경, 성산일출봉의 일출, 겨울바다 등을 즐길 수 있습니다. 

4. 강원도 속초: 겨울바다를 즐기고 싶다면 속초가 좋습니다. 속초해수욕장, 아바이마을, 청초호 등에서 겨울바다를 만끽할 수 있습니다.

5. 충북 단양: 소백산의 겨울 풍경, 고수동굴, 천동동굴 등에서 겨울 자연을 만끽할 수 있습니다. 

6. 경기도 포천: 아트밸리나 허브아일랜드 등에서 겨울에도 다양한 문화 체험을 즐길 수 있습니다.

7. 전라북도 전주: 한옥마을의 겨울 풍경, 전동성당, 남부시장 등에서 겨울의 전주를 즐길 수 있습니다. 

8. 경상남도 통영: 동피랑 벽화마을, 중앙시장, 미륵산 등에서 겨울의 통영을 즐길 수 있습니다. 

9. 충남 보령: 대천해수욕장의 겨울 바다, 무창포 등에서 겨울의 보령을 즐길 수 있습니다.

10. 경기도 가평: 자라섬, 남이섬 등에서 겨울의 가평을 즐길 수 있습니다.

=== temperature=1.0 ===
1. 휘닉스 평창: 볼거리, 먹거리, 즐길거리가 한 곳에 모인 곳으로 스키나 눈썰매 등 겨울 스포츠를 즐길 수 있으며, 새해맞이 행사도 열린다.

2. 허브 아일랜드: 겨울철에도 다양한 허브와 꽃이 만개해 있고, 따뜻한 실내식물원에서는 허브차를 마시며 휴식을 취할 수 있다.

3. 대관령 삼양목장: '제2의 설악산'으로 불리는 곳으로 겨울철에는 설경이 더욱 아름답다. 또한, 삼양목장 내에 위치한 카페에서는 목장의 경치를 즐기며 따뜻한 커피를 마실

In [ ]:
#파라미터 조절로 답변 스타일 변화 보기

resp_low = client.responses.create(
    model=MODEL,
    input="",
    temperature=0.2,
)
resp_high = client.responses.create(
    model=MODEL,
    input="",
    temperature=1.0,
)

print("=== temperature=0.2 ===")
print(resp_low.output_text)
print("\n=== temperature=1.0 ===")
print(resp_high.output_text)

In [ ]:
#구조화된 출력(JSON) 받기
prompt = """
Return JSON only with keys:
- definition (string)
- examples (list of 2 strings)
- gotchas (list of 2 strings)
Topic: Python isinstance()
"""

resp = client.responses.create(
    model=MODEL,
    input=prompt,
)

print(resp.output_text)  # JSON처럼 보이는 텍스트가 나와야 함

{
  "definition": "isinstance() is a built-in Python function for checking if an object is an instance or subclass of a specified class.",
  "examples": [
    "Example 1: isinstance(3, int) would return True because 3 is an instance of the int class.",
    "Example 2: isinstance('Hello', str) would return True because 'Hello' is an instance of the str class."
  ],
  "gotchas": [
    "Gotcha 1: isinstance() can sometimes return unexpected results when dealing with Python's built-in types, because they can often behave like subclasses of each other.",
    "Gotcha 2: issinstance() does not consider the value, it only checks the type. For instance, isinstance(\"3\", int) will return False, because \"3\" is a string, not an integer."
  ]
}


한국어

In [ ]:
import os
from openai import OpenAI

MODEL = "gpt-4-0613"

resp = client.responses.create(
    model=MODEL,
    instructions="너는 간결하고 정확한 파이썬 코딩 튜터야. 한국어로 답해.",
    input="파이썬에서 어떤 객체가 특정 클래스의 인스턴스인지 확인하는 방법을 예제와 함께 설명해줘.",
)

print(resp.output_text)


파이썬에서 객체가 특정 클래스의 인스턴스인지를 확인하는 방법은 내장 함수 `isinstance()`를 사용하는 것입니다. 

예제를 통해 설명하겠습니다.

```python
class MyClass:
  pass

my_instance = MyClass()

print(isinstance(my_instance, MyClass))  # 이 코드은 True를 출력합니다.
```

위 예제에서, `MyClass`는 우리가 정의한 클래스이고, `my_instance`는 `MyClass`의 인스턴스입니다.

`isinstance()` 함수는 첫 번째 인자로 인스턴스, 두 번째 인자로 클래스를 받습니다. 첫 번째 인자가 두 번째 인자의 인스턴스라면 `True`를 그렇지 않다면 `False`를 반환합니다. 따라서 코드 `isinstance(my_instance, MyClass)`는 `my_instance`가 `MyClass`의 인스턴스인지를 검사하고, 그 결과가 `True`이므로 `True`를 출력합니다.


In [ ]:
messages = [
    {"role": "system", "content": "너는 간결한 파이썬 코딩 튜터야. 한국어로만 답해."},
    {"role": "user", "content": "isinstance()를 3줄로 설명해줘."},
    {"role": "user", "content": "상속 관계에서 주의할 점도 2개만 추가해줘."},
]

resp = client.responses.create(model=MODEL, input=messages)
print(resp.output_text)

1. 부모 클래스에서 정의된 메서드를 자식 클래스에서 재정의(오버라이딩) 할 때는 반드시 메서드의 인자를 동일하게 맞춰야 합니다. 인자를 변경하면 문제가 발생할 수 있습니다.
2. 메서드 오버라이딩을 할 때, 자식 클래스의 메서드에서 super() 함수를 사용해 부모 클래스의 메서드를 호출하지 않으면, 부모 클래스의 행동이 완전히 무시되므로 원하는 결과를 얻지 못할 수 있습니다.


In [ ]:
prompt = """
아래 키를 갖는 JSON만 출력해줘.
- 정의: string
- 예시: string 2개 배열
- 주의점: string 2개 배열

주제: 파이썬 isinstance()
"""

resp = client.responses.create(model=MODEL, input=prompt)
print(resp.output_text)

{
    "정의": "isinstance()는 파이썬 내장 함수로, 첫 번째 인자로 전달된 객체가 두 번째 인자로 전달된 클래스의 인스턴스이거나 서브 클래스인지 판별합니다.",
    "예시": ["isinstance(5, int)", "isinstance('hello', str)"],
    "주의점": ["isinstance는 인스턴스 유형만 검사하며, 값을 검사하지 않습니다.", "isinstance는 상속 관계도 고려하므로 서브 클래스의 인스턴스도 참으로 판별합니다."]
}


# ChatOpenAI 주요 매개변수 & 출력 실습

이 노트북에서는 LangChain의 `ChatOpenAI`를 사용해 다음을 실습합니다.

- `model`, `temperature`, `max_tokens`, `stop` 등 핵심 매개변수 체감
- 응답 객체(`AIMessage`)에서 `content`, `usage_metadata`, `response_metadata` 확인
- 스트리밍 출력(`stream`) 사용
- 구조화 출력(`with_structured_output`)로 파싱 가능한 결과 받기



In [ ]:
!pip -q install -U langchain-openai langchain-core pydantic

In [ ]:
import os, getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ").strip()

# 이후부터는 같은 세션에서 getpass를 다시 묻지 않습니다.

OPENAI_API_KEY: ··········


- 클라이언트 생성+모델 목록 확인

In [ ]:
from openai import OpenAI
client = OpenAI()

models = [m.id for m in client.models.list().data]
models[:20], len(models)

(['gpt-4-0613',
  'gpt-4',
  'gpt-3.5-turbo',
  'chatgpt-image-latest',
  'gpt-4o-mini-tts-2025-03-20',
  'gpt-4o-mini-tts-2025-12-15',
  'gpt-realtime-mini-2025-12-15',
  'gpt-audio-mini-2025-12-15',
  'davinci-002',
  'babbage-002',
  'gpt-3.5-turbo-instruct',
  'gpt-3.5-turbo-instruct-0914',
  'dall-e-3',
  'dall-e-2',
  'gpt-4-1106-preview',
  'gpt-3.5-turbo-1106',
  'tts-1-hd',
  'tts-1-1106',
  'tts-1-hd-1106',
  'text-embedding-3-small'],
 114)

- temperature: 0~2. 높을수록 랜덤/창의, 낮을수록 결정적. 보통 top_p와 둘 중 하나만 조정 권장.

- top_p: 누적확률(top-p) 기반 샘플링. temperature와 둘 중 하나만 조정 권장.

- stop: 최대 4개 정지 시퀀스. 결과 텍스트에 stop 문자열은 포함되지 않음.

- presence_penalty, frequency_penalty: 반복/새 주제 유도에 영향.

- seed: 가능하면 재현성(완전 동일 보장은 아님).

- streaming: 토큰을 생성되는 대로 스트리밍. (LangChain에서 콜백으로 받는 방식)

    - 참고: LangChain ChatOpenAI(...)는 공통 옵션(예: timeout, max_retries, streaming)과 더불어, OpenAI에 전달되는 추가 파라미터를 함께 사용할 수 있습니다.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

#temperature 비교 실습
prompt = "Give me 5 short titles for a talk on quantum computing for AI-savvy mathematicians."

for t in [0.0, 0.3, 0.9]:
    llm_t = ChatOpenAI(model="gpt-4-0613", temperature=t)
    out = llm_t.invoke(prompt)
    print(f"\n--- temperature={t} ---")
    print(out.content)



--- temperature=0.0 ---
1. "Harnessing Quantum Computing for Advanced AI Algorithms"
2. "The Intersection of Quantum Computing and AI: A Mathematical Perspective"
3. "Quantum Computing: A New Frontier in AI Mathematics"
4. "Exploring Quantum Algorithms for AI: A Mathematical Approach"
5. "The Role of Quantum Computing in the Evolution of AI Mathematics"

--- temperature=0.3 ---
1. "Harnessing Quantum Computing for Advanced AI Algorithms"
2. "Quantum Computing: A New Frontier in AI Mathematics"
3. "Exploring Quantum Algorithms for AI Optimization"
4. "The Intersection of Quantum Computing and AI: A Mathematical Perspective"
5. "Quantum Computing: Revolutionizing AI and Mathematical Computation"

--- temperature=0.9 ---
1. "Envisioning AI’s Future: The Quantum Computing Revolution"
2. "Quantum Computing: The Next Frontier in AI"
3. "An Intersection of Mathematics and AI: Quantum Computing"
4. "Expanding AI Horizons: A Deep Dive into Quantum Computing"
5. "Quantum Computing: Unveiling Ne

In [ ]:
# top_p 비교 실습
for p in [1.0, 0.3, 0.1]:
    llm_p = ChatOpenAI(model="gpt-4-0613", temperature=1.0, top_p=p)
    out = llm_p.invoke("Write 3 different one-sentence metaphors for entanglement.")
    print(f"\n--- top_p={p} ---")
    print(out.content)



--- top_p=1.0 ---
1. Entanglement is like the invisible thread of destiny, binding distant entities in an intricately woven dance of cause and effect.
2. It’s the mysterious echo in a vast canyon, where the whisperings of one particle reverberate seamlessly in another, confounding the canyon of space-time.
3. Entanglement is akin to a symphony where the music notes, though miles apart, harmonize flawlessly in an inexplicable cosmic concert.

--- top_p=0.3 ---
1. Entanglement is like a cosmic dance where two particles, no matter how far apart, move in perfect synchrony.
2. It's a mysterious thread, invisible and unbreakable, connecting two entities across the vast expanse of space.
3. Entanglement is the secret whisper of the universe, a message passed instantly between two particles, defying the laws of time and space.

--- top_p=0.1 ---
1. Entanglement is like a cosmic dance where two particles, no matter how far apart, move in perfect synchrony.
2. It's a mysterious thread that invi

In [29]:
from openai import OpenAI
from openai import BadRequestError, AuthenticationError, RateLimitError, APIError


PROMPT = (
    "너는 고등학생에게 설명하는 선생님이야.\n"
    "주제: 피타고라스 정리\n\n"
    "아래 형식을 반드시 지켜서 답해줘.\n"
    "(1) 핵심 3줄(불릿)\n"
    "(2) 숫자 예시 1개 (계산 포함)\n"
    "(3) 마지막에 퀴즈 1문제(정답은 쓰지 말기)\n"
)

cases = [
    ("A) temperature 낮게(정돈)", dict(temperature=0.1, max_output_tokens=220)),
    ("B) temperature 높게(다양)", dict(temperature=1.2, max_output_tokens=220)),
    ("C) top_p 낮게(보수)",       dict(top_p=0.2, max_output_tokens=220)),
    ("D) 짧게 끊기",              dict(temperature=0.7, max_output_tokens=90)),
]

MODEL = "gpt-5.2"

for title, params in cases:
    print("\n" + "=" * 80)
    print(title, params)
    try:
        r = client.responses.create(model=MODEL, input=PROMPT, **params)
        print(r.output_text)
    except AuthenticationError as e:
        print("[인증 오류] API Key가 잘못됐거나 미설정:", e)
        break
    except BadRequestError as e:
        print("[요청 오류] 모델/파라미터 조합이 지원되지 않을 수 있음:", e)
    except RateLimitError as e:
        print("[레이트리밋] 잠시 후 재시도:", e)
    except APIError as e:
        print("[API 오류] 일시적 장애 가능:", e)


A) temperature 낮게(정돈) {'temperature': 0.1, 'max_output_tokens': 220}
(1) 핵심 3줄(불릿)  
- 직각삼각형에서 빗변의 제곱은 나머지 두 변(직각을 이루는 변) 제곱의 합과 같다.  
- 식으로는 \(a^2 + b^2 = c^2\) (여기서 \(c\)는 빗변, \(a,b\)는 직각변).  
- 두 변의 길이를 알면 나머지 한 변의 길이를 구할 때 매우 유용하다.

(2) 숫자 예시 1개 (계산 포함)  
직각삼각형에서 직각변이 \(a=3\), \(b=4\)일 때 빗변 \(c\)를 구해보자.  
\[
a^2 + b^2 = c^2
\]
\[
3^2 + 4^2 = c^2
\]
\[
9 + 16 = c^2
\]
\

B) temperature 높게(다양) {'temperature': 1.2, 'max_output_tokens': 220}
(1) 핵심 3줄(불릿)  
- 직각삼각형에서 빗변의 제곱은 다른 두 변(직각을 이루는 변) 제곱의 합이다.  
- 식으로는 \(a^2 + b^2 = c^2\) (여기서 \(c\)는 빗변, \(a,b\)는 직각변).  
- 두 변의 길이를 알면 나머지 한 변의 길이를 제곱과 제곱근으로 구할 수 있다.

(2) 숫자 예시 1개 (계산 포함)  
직각삼각형에서 직각변이 \(a=3\), \(b=4\)일 때 빗변 \(c\)를 구해보자.  
\[
a^2 + b^2 = c^2
\]
\[
3^2 + 4^2 = c^2
\]
\[
9 + 16 = c^2


C) top_p 낮게(보수) {'top_p': 0.2, 'max_output_tokens': 220}
(1) 핵심 3줄(불릿)
- 직각삼각형에서 빗변의 제곱은 나머지 두 변(직각을 이루는 변) 제곱의 합과 같다.  
- 식으로는 \(a^2 + b^2 = c^2\) (여기서 \(c\)는 빗변, \(a,b\)는 직각변).  
- 두 변의 길이를 알면 나머지 한 변의 길이를 구할 때 매우 자주 쓰인다.

(2) 숫자 예시 1개 (계산 포함)
-

In [ ]:
# stop 실습(원하는 지점에서 강제 종료)

llm_stop = ChatOpenAI(model="gpt-4-0613", temperature=0.2, stop=["\n\n"])
out = llm_stop.invoke("List 10 keywords about Bell states, one per line.\n\n(End)")
print(out.content)

Quantum Entanglement
Quantum Mechanics
Quantum Superposition
Quantum Information Theory
Quantum Computing
Quantum Teleportation
Quantum Cryptography
Quantum Bits (Qubits)
Bell's Theorem
EPR Paradox


In [ ]:
# 반복 어제/새 주제 유도(penalty) 실습
base = "Write a 120-word explanation of Bell states. Avoid repeating the same phrases."

settings = [
    {"presence_penalty": 0.0, "frequency_penalty": 0.0},
    {"presence_penalty": 0.8, "frequency_penalty": 0.0},
    {"presence_penalty": 0.0, "frequency_penalty": 0.8},
]

for s in settings:
    llm_pen = ChatOpenAI(model="gpt-4-0613", temperature=0.6, **s)
    out = llm_pen.invoke(base)
    print(f"\n--- {s} ---")
    print(out.content)



--- {'presence_penalty': 0.0, 'frequency_penalty': 0.0} ---
Bell states, named after physicist John Bell, are specific quantum states of two qubits that represent the simplest examples of quantum entanglement. Quantum entanglement is a phenomenon where particles become interconnected, with the state of one instantly influencing the other, regardless of the distance separating them. The four Bell states are maximally entangled quantum states, meaning measurements on one qubit immediately alter the state of the other. These states play a crucial role in quantum information science, including quantum computing and quantum teleportation. Understanding Bell states is fundamental to grasping the unique properties of quantum mechanics.

--- {'presence_penalty': 0.8, 'frequency_penalty': 0.0} ---
Bell states, named after physicist John Bell, are specific quantum states of two qubits that represent the simplest examples of quantum entanglement. Quantum entanglement is a phenomenon where partic

In [ ]:
# seed 실습(가능하면 재현성 높이기)

llm_seed1 = ChatOpenAI(model="gpt-4-0613", temperature=0.7, seed=123)
llm_seed2 = ChatOpenAI(model="gpt-4-0613", temperature=0.7, seed=123)

a = llm_seed1.invoke("Give 3 creative analogies for quantum superposition.")
b = llm_seed2.invoke("Give 3 creative analogies for quantum superposition.")

print("A:\n", a.content)
print("\nB:\n", b.content)


A:
 1. Imagine a spinning coin. When it's in the air, spinning rapidly, it's neither heads nor tails, but a blur of both. That's a superposition. It's only when the coin lands and stops spinning that it becomes either heads or tails. Similarly, in quantum physics, a particle can be in a superposition of states until it is measured or observed, at which point it collapses into one state.

2. Quantum superposition is like a radio playing all stations at once. The sound you hear is a cacophony of every station broadcasted, all layered on top of each other. When you turn the tuning knob, you're making a measurement, collapsing the superposition into one frequency. 

3. Think of a book with multiple endings. As you read the book, all endings are possible and exist simultaneously. However, once you reach the end of the story, only one ending is realized, and all other endings collapse, much like the quantum superposition.

B:
 1. Imagine a spinning coin. When it's in the air, it's neither he

-“출력(결과 객체)” 확인 실습 (AIMessage 내부 보기)

    - LangChain 반환값은 보통 AIMessage이고, 아래 속성이 자주 유용합니다:

    - content: 모델 답변 텍스트

    - response_metadata: 모델/응답 메타데이터(모델명, 종료사유 등)

    - usage_metadata: 토큰 사용량(환경/버전에 따라 형태가 다를 수 있음)

In [ ]:
from pprint import pprint

llm = ChatOpenAI(model="gpt-4-0613", temperature=0.2)
res = llm.invoke("In 2 bullet points, explain why entanglement is hard to scale in hardware.")

print("content:\n", res.content)

print("\nresponse_metadata:")
pprint(getattr(res, "response_metadata", None))

print("\nusage_metadata:")
pprint(getattr(res, "usage_metadata", None))

print("\nfull object dict (keys only):")
print(list(res.__dict__.keys()))

content:
 - Maintaining entanglement: As the number of entangled particles increases, it becomes exponentially more difficult to maintain their entangled state. Any interaction with the environment (known as decoherence) can cause the particles to lose their entanglement.

- Complexity of operations: The number of operations needed to manipulate and control a large number of entangled particles grows exponentially with the number of particles. This requires a significant increase in computational resources and precision, making it challenging to scale up.

response_metadata:
{'finish_reason': 'stop',
 'id': 'chatcmpl-Ct3qib5TSqGWyCe0Y2zDKwyje7yYl',
 'logprobs': None,
 'model_name': 'gpt-4-0613',
 'model_provider': 'openai',
 'service_tier': 'default',
 'system_fingerprint': None,
 'token_usage': {'completion_tokens': 97,
                 'completion_tokens_details': {'accepted_prediction_tokens': 0,
                                               'audio_tokens': 0,
                     

- 스트리밍 출력 실습

    - 스트리밍은 OpenAI에서 stream=true로 토큰을 순차 전송하는 방식입니다.
    - LangChain에서는 콜백 핸들러로 토큰을 받습니다.

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler

class PrintTokens(BaseCallbackHandler):
    def on_llm_new_token(self, token: str, **kwargs):
        print(token, end="", flush=True)

llm_stream = ChatOpenAI(
    model="gpt-4-0613",
    temperature=0.4,
    streaming=True,
    callbacks=[PrintTokens()],
)

_ = llm_stream.invoke("Explain Bell states in a friendly way for AI engineers, 6-8 sentences.")
print()  # 줄바꿈

Bell states, named after physicist John Bell, are specific quantum states of two qubits that represent the simplest examples of quantum entanglement. Quantum entanglement is a phenomenon where two particles become linked and the state of one instantly influences the state of the other, no matter the distance between them. 

In the context of quantum computing, Bell states are important because they provide a fundamental resource for quantum information processes such as quantum teleportation and superdense coding. Understanding and utilizing Bell states is key to developing and optimizing quantum algorithms and applications. Essentially, they represent the basic building blocks of quantum information science, much like how binary states are fundamental in classical computing.


- 실습용 “템플릿” 함수: 파라미터 바꿔가며 바로 비교

In [ ]:
def run(prompt, **kwargs):
    llm = ChatOpenAI(model="gpt-4-0613", **kwargs)
    res = llm.invoke(prompt)
    return res.content

prompt = "Write a 90-word intro slide text about quantum computers (for AI-savvy mathematicians)."

print(run(prompt, temperature=0.2))
print("\n---\n")
print(run(prompt, temperature=0.8))

Welcome to the fascinating world of quantum computing. As AI-savvy mathematicians, you're aware of the power of algorithms and computations. Now, imagine that power exponentially increased. Quantum computers leverage the principles of quantum mechanics to process information in ways classical computers cannot, promising unprecedented computational speed and capacity. They hold the potential to revolutionize AI, cryptography, and complex problem-solving. Let's delve into the quantum realm and explore its implications for the future of computing.

---

Welcome to the captivating world of Quantum Computers. Utilizing principles of quantum mechanics, these cutting-edge machines can process complex computations at speeds inconceivable today. This exciting technology intersects perfectly with the realm of Artificial Intelligence, opening up possibilities for enhanced machine learning, optimization and data handling. As mathematicians, understanding quantum computing can empower you to tap in